In [2]:
import numpy as np
import matplotlib.pyplot as plt
import os
import sys
sys.path.insert(0, "/home/thomasb/")
from matplotlib.gridspec import GridSpec


from albatros_analysis.src.utils import orbcomm_utils as outils

In [3]:
coords = {
        #0: [79.417161473, -90.767238685, 187.9577], #MARS1
        0: [79.417198047, -90.758739192, 183.0684], #MARS2
        1: [79.388456412, -91.019202963, 25.1938], #CSA
        2: [79.418302573, -90.667395452, 59.6242], #Mars 5
        3: [79.397984238, -90.799842408, 41.6994], #Mars 6
        4: [79.411474117, -90.695266129, 31.6314], #Mars 7
        5: [79.443757694, -90.718202634, 414.9131] #MARS8
    }
#antmap = {0:"MARS1", 1:"MARS2",2:"MARS4",3:"MARS5",4:"MARS6",5:"MARS7",6:"MARS8"}
antmap = {0:"MARS2", 1:"MARS4",2:"MARS5",3:"MARS6",4:"MARS7",5:"MARS8"}

# Automate Everything

In [ ]:
#find a way to spit out the satellite automatically. this is for each pulse in batch 1762099360
satlist = [57166, 59051, 57166, 59051, 57166, 59051, 57166, 57166, 59051, 57166, 59051, 57166, 59051, 59051, 59051, 59051, 59051, 57166]
print(len(satlist))

18


In [ ]:
for pidx in range(len(satlist)):

    fname = f'/scratch/thomasb/pipeline_test_phases2/data_without_clock/satpass_{pidx}.npz'

    path_plots = f'/scratch/thomasb/pipeline_test_phases2/satpass_{pidx}_plots'

    os.makedirs(path_plots, exist_ok = True)

    with np.load(fname) as f:
        print(f)
        vis = f['data']
        times = f['times']
        freqs = f['freqs']

    ntimes, nchans, nbl = vis.shape
    print(vis.shape)
    triu_idx=np.triu_indices(6,k=1)
    tle_path = outils.get_tle_file(times[0], "/project/rrg-sievers/mohanagr/OCOMM_TLES")
    sat = satlist[pidx]
    chanid = 10

    print('vis shape', vis.shape)
    print('utc pass shape', times.shape)
    print('start of pass', times[0])
    print('dt', times[1]-times[0])
    print('freqs shape', freqs.shape)
    print('start freq', freqs[0])
    print('df', freqs[1]-freqs[0])

    for blid in range(nbl):

        ant1_idx, ant2_idx = triu_idx[0][blid], triu_idx[1][blid]
        name1, name2 = antmap[ant1_idx], antmap[ant2_idx]
        coord1, coord2 = coords[ant1_idx], coords[ant2_idx]

        freq = freqs[chanid]

        print(ant1_idx, ant2_idx)
        print(name1, name2)
        print('Frequency', freq / 1e6, 'MHz')

        # ============================================================
        # Measured phase

        angle_meas = np.angle(vis[:, chanid, blid])
        phase_meas = np.unwrap(angle_meas)
        phase_meas -= phase_meas[0]

        # ============================================================
        # Predicted phase stuff

        niter = int(times[-1] - times[0]) + 2
        dt = times[1] - times[0]

        d = outils.get_sat_delay(coord1,coord2,tle_path,times[0],niter,sat)
        delays = np.interp(np.arange(ntimes) * dt, np.arange(niter), d)

        angle_pred = -2 * np.pi * freq * delays
        phase_pred = angle_pred - angle_pred[0]

        residuals = phase_meas - phase_pred
        phase_gradient = np.gradient(phase_pred, times)

        # ============================================================
        # Plot

        fig, (ax1, ax2, ax3, ax4) = plt.subplots(4, 1,figsize=(12, 12),sharex=True,gridspec_kw={'height_ratios': [2, 1, 2.5, 1.5],'hspace': 0.08})
        tplot = times - times[0]

        # ============================================================
        # 1. Predicted vs actual phase

        ax1.plot(tplot,phase_pred,label=f'Predicted ({sat})',linewidth=1.5)
        ax1.plot(tplot,phase_meas,label='Measured',linewidth=1)
        ax1.set_ylabel('Phase (rad)')
        ax1.legend()

        # ============================================================
        # 2. Residuals

        ax2.plot(tplot, residuals,linewidth=1)
        ax2.axhline(0,linestyle='--',linewidth=1,alpha=0.5,color='black')
        ax2.set_ylabel('Residual (rad)')

        # ============================================================
        # 3. Visibility phase
        # ============================================================

        phase_image = np.angle(vis[:, :, blid])
        ax3.imshow(phase_image.T,aspect='auto',interpolation='none',cmap='RdBu',origin='lower',vmin=-np.pi,vmax=np.pi,extent=[tplot[0],tplot[-1],0,len(freqs)])

        # Mark the channel used for the phase comparison
        ax3.axhline(chanid, linestyle='--', linewidth=1)
        ax3.set_ylabel('Frequency (MHz)')
        ax3.set_ylim(chanstart, chanend)

        # ============================================================
        # 4. Predicted phase gradient

        ax4.plot(tplot, phase_gradient, linewidth=1.5, label='Gradient')

        ax4.axhline(0,linestyle='-',linewidth=1,alpha=0.5,color='black')

        unwrap_limit = np.pi / dt

        ax4.axhline(unwrap_limit,linestyle=':',linewidth=1,color='r',label=r'$\pm\pi/\Delta t$')
        ax4.axhline(-unwrap_limit, linestyle=':', linewidth=1, color='r')
        ax4.set_xlim(tplot[0], tplot[-1])
        ax4.set_xlabel('Time since start (s)')
        ax4.set_ylabel(r'$d\phi/dt$' + '\n(rad/s)')
        ax4.legend()

        # ============================================================
        fig.suptitle(f'Baseline {name1}-{name2} (id {blid}) Channel id {chanid}',fontsize=14, y=0.98)

        fig.tight_layout()

        path_plot = os.path.join(path_plots, f'{name1}_{name2}.png')
        fig.savefig(path_plot, dpi=150, bbox_inches='tight')
        plt.close(fig)

NpzFile '/scratch/thomasb/pipeline_test_phases2/data_without_clock/satpass_0.npz' with keys: data, mask, times, freqs
(1964, 72, 15)
0 1
MARS2 MARS4
Frequency 137.908935546875 MHz
catalog #57166 epoch 2025-11-02 12:01:12 UTC
one delay prediction: 0.06297922134399414


/tmp/ipykernel_1763706/1978683089.py:131: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  fig.tight_layout()


0 2
MARS2 MARS5
Frequency 137.908935546875 MHz
catalog #57166 epoch 2025-11-02 12:01:12 UTC
one delay prediction: 0.060045719146728516
0 3
MARS2 MARS6
Frequency 137.908935546875 MHz
catalog #57166 epoch 2025-11-02 12:01:12 UTC
one delay prediction: 0.05467629432678223
0 4
MARS2 MARS7
Frequency 137.908935546875 MHz
catalog #57166 epoch 2025-11-02 12:01:12 UTC
one delay prediction: 0.05794477462768555
0 5
MARS2 MARS8
Frequency 137.908935546875 MHz
catalog #57166 epoch 2025-11-02 12:01:12 UTC
one delay prediction: 0.061608076095581055
1 2
MARS4 MARS5
Frequency 137.908935546875 MHz
catalog #57166 epoch 2025-11-02 12:01:12 UTC
one delay prediction: 0.06810832023620605
1 3
MARS4 MARS6
Frequency 137.908935546875 MHz
catalog #57166 epoch 2025-11-02 12:01:12 UTC
one delay prediction: 0.060468196868896484
1 4
MARS4 MARS7
Frequency 137.908935546875 MHz
catalog #57166 epoch 2025-11-02 12:01:12 UTC
one delay prediction: 0.058289289474487305
1 5
MARS4 MARS8
Frequency 137.908935546875 MHz
catalog #57